# SOLUTIONS · Laptop lab 2: Nearest neighbors and linear regression
*Choose a model you can trust on new data*

**Machine Learning with Python** · Minor Amsterdam Data Science and Artificial Intelligence · Fall 2026<br>
Thursday, November 5, 2026 · Laptop labs: Batuhan Ataş (b.atas@uva.nl)

> **Before you start:** choose **File → Save a copy in Drive** and work in that copy. Colab does not save your changes in the original notebook.

This week you met your first models: $k$-nearest neighbors (Lecture 3), and linear regression with ridge and lasso (Lecture 4). Lecture 3 also gave a warning: a model is only as good as its score on data it has *never seen*, and you must not choose a model by looking at the test set. Today you feel why.

**The story.** You get the prices of 500 Airbnb listings in Amsterdam, and your job is to find the model that predicts the price per night best. But 1,500 other listings stay locked away: the **test set**. At the end of the lab the tutor unlocks it, and every model in the room is judged on listings it has never seen.

**Your mission.** Choose one model, with one setting, and estimate how well it will do on the locked listings:

- $k$-NN regression, with your choice of $k$ (Lecture 3);
- ridge regression or lasso, with your choice of `alpha` (Lecture 4).

*How* you choose is up to you: by the score on the training data, with one validation split, with cross-validation and any number of folds, with a grid search. Different people will choose differently, and that is the point: at the end we compare.

This lab is different from Lab 1: fewer explanations, and more of your own work. The warm-up shows every step once; after that, you decide.

## How this lab works

| You see | What you do |
|---|---|
| **Run** | Run the cell and look at the output. |
| **Your turn** | Write the code yourself, or fill in the dots `...`. The cell shows an orange *Not yet* box until you do. |
| **Toolbox** | Click to unfold a piece of code that you can copy and change. |
| **Think** | Double-click *Your answer* and write a sentence or two. We discuss the answers together. |
| **Stop** | Wait here until the tutor says you can go on. |

Every model in scikit-learn works the same way (Lecture 3):

```python
model = SomeModel(setting=...)   # choose the model and its hyperparameter
model.fit(X_train, y_train)      # learn from the training data
model.predict(X_new)             # predictions for new listings
model.score(X_new, y_new)        # for regression: R², 1 is perfect, 0 is no better than the mean
```

| Part | Minutes |
|---|---|
| 0. Setup | 5 |
| 1. Warm-up: one model, start to finish | 15 |
| 2. Your mission | 35 |
| 3. The test set is unlocked | 10 |
| 4. What happened? | together, 15 |
| Check your understanding | together, at the end |

## Task 0. Setup

Run the cell below. It loads the libraries, downloads the tools of this lab and the data from Inside Airbnb, and cleans them as in Lab 1. You should see a green box with a ✓.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Ridge, Lasso

# The tools of this lab are in the course lab repository;
# they also download the data from Inside Airbnb.
BASE = "https://raw.githubusercontent.com/batuatas/MLwPython-labs/main/"
import urllib.request
urllib.request.urlretrieve(BASE + "lab02_tools.py", "lab02_tools.py")
from lab02_tools import *

X, y = load_training_data()    # your 500 listings: features X, prices y
setup_ok()

**Run.** Your data: one row per listing. `y` holds the price per night in euros, the label. `X` holds thirteen features:

| Feature | Meaning |
|---|---|
| `accommodates`, `bedrooms`, `beds` | maximum number of guests, number of bedrooms and beds |
| `entire_home` | 1 for a whole home, 0 for a room |
| `latitude`, `longitude` | location |
| `km_to_center` | distance to Dam Square, the center of Amsterdam, in km. In Lab 1, prices were highest in the middle of the city and dropped towards the edges, a pattern that a straight line in latitude and longitude cannot follow; this feature lets a linear model see it. Making new features from what you know about the data is the topic of Lecture 6. |
| `minimum_nights`, `availability_365` | shortest stay allowed, nights still bookable in the coming year |
| `number_of_reviews`, `review_scores_rating` | number of reviews, average rating |
| `host_listings`, `superhost` | listings of the host in Amsterdam, 1 if the host is a superhost |

Lecture 3 showed that the distances of $k$-NN depend on the units: a feature in minutes would outweigh one in hours. So we have put all features on one scale for you: each feature has mean 0 and standard deviation 1. A value of 2 means "two standard deviations above the average listing". Lecture 6 shows how to do this yourself. The original values are in `load_training_listings()`.

In [ ]:
print(X.shape, y.shape)
X.head()

## Task 1. Warm-up: one model, start to finish

Before the mission, here is every step once, with one model: $k$-NN with $k = 5$. Read the code; you will use the same lines in your mission.

**Run.** First a *validation split*, as in Lecture 3: we keep a quarter of the training listings apart, train on the rest, and score on the part the model did not see. The test set stays locked: the validation set is our own, smaller test inside the training data.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=0)

model = KNeighborsRegressor(n_neighbors=5)
model.fit(X_train, y_train)                   # learn from 375 listings
print("R² on the training part:  ", round(model.score(X_train, y_train), 2))
print("R² on the validation part:", round(model.score(X_val, y_val), 2))

**Think.** Why is the score on the training part higher than on the validation part? Which of the two tells you more about new listings?

**Answer.** On the training part, every listing is among its own five nearest neighbors, so the model partly remembers the answers. The validation listings are new to the model, so their score tells you how it does on listings it has not seen. That is the number that matters (Lecture 3, "Training data and test data").

**Run.** One split is one draw: other listings in the validation part would give another score. *Cross-validation* (Lecture 3) repeats the split: with `cv=5`, every listing is in the validation part exactly once, and you get five scores.

In [ ]:
scores = cross_val_score(KNeighborsRegressor(n_neighbors=5), X, y, cv=5)
print("five validation scores:", scores.round(2))
print("their mean:", round(scores.mean(), 2))

**Think.** The five scores differ. Why? And which estimate would you trust more for new listings: the one split above, or the mean of the five?

**Answer.** Each fold holds other listings, some easier to predict than others, so the scores vary. The mean of five folds uses every listing once for validation, so it depends less on one lucky or unlucky split: a more reliable estimate (Lecture 3, "Cross-validation").

## Task 2. Your mission

Now it is your turn. Choose **one model with one setting**, and estimate its $R^2$ on the locked listings.

- **The models:** `KNeighborsRegressor(n_neighbors=k)`, `Ridge(alpha=...)` or `Lasso(alpha=...)`.
- **The settings:** for $k$, try values from 1 to 50 or so. For `alpha`, try very different values, such as 0.1, 1, 10, 100 and 1000: the right `alpha` can be small or large.
- **The rule:** you may do anything with `X` and `y`. The test set stays locked.
- **The choice of strategy is yours.** Choose by the score on the training data, with one validation split (how large?), with cross-validation (how many folds?), or with a grid search. Unfold the toolbox below for code that you can copy and change.

Write your experiments in the cells below; add cells if you need more. Keep a note of what you tried and what you found.

<details><summary><b>Toolbox</b> (click to unfold)</summary>

**A. The score on the training data.** Fit on all training listings, and score on the same listings.

```python
for k in [1, 5, 10, 20, 50]:
    model = KNeighborsRegressor(n_neighbors=k).fit(X, y)
    print(k, round(model.score(X, y), 3))
```

**B. One validation split.** As in the warm-up. Choose the size of the validation part with `test_size`.

```python
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=0)
for alpha in [0.1, 1, 10, 100, 1000]:
    model = Ridge(alpha=alpha).fit(X_train, y_train)
    print(alpha, round(model.score(X_val, y_val), 3))
```

**C. Cross-validation.** Choose the number of folds with `cv`.

```python
for k in [1, 5, 10, 20, 50]:
    scores = cross_val_score(KNeighborsRegressor(n_neighbors=k), X, y, cv=5)
    print(k, round(scores.mean(), 3))
```

**D. Grid search** (Lecture 3, "Grid search: try every $k$"): cross-validation for every value in the grid, in one line.

```python
grid = GridSearchCV(Lasso(), {"alpha": [0.1, 1, 10, 100]}, cv=5)
grid.fit(X, y)
print(grid.best_params_, round(grid.best_score_, 3))
```

**A plot** of the scores against $k$ (or against `alpha`) helps you see the sweet spot of Lecture 3:

```python
ks = range(1, 51)
means = [cross_val_score(KNeighborsRegressor(n_neighbors=k), X, y, cv=5).mean() for k in ks]
fig, ax = plt.subplots()
ax.plot(ks, means)
ax.set_xlabel("k")
ax.set_ylabel("mean validation R²")
```

</details>

In [ ]:
# Our choice: compare the three models by 5-fold cross-validation, with a grid search each.
grids = {
    "k-NN": GridSearchCV(KNeighborsRegressor(), {"n_neighbors": list(range(1, 51))}, cv=5, return_train_score=True),
    "Ridge": GridSearchCV(Ridge(), {"alpha": [0.01, 0.1, 1, 10, 30, 100, 300, 1000, 3000, 10000]}, cv=5, return_train_score=True),
    "Lasso": GridSearchCV(Lasso(), {"alpha": [0.01, 0.1, 1, 3, 10, 30, 100, 300]}, cv=5, return_train_score=True),
}
for name, grid in grids.items():
    grid.fit(X, y)
    print(f"{name:6s} best setting {grid.best_params_}, mean validation R² {grid.best_score_:.3f}")

In [ ]:
# Inside each grid search: the training and the validation R² for every setting.
# (scikit-learn calls the validation folds "test" in cv_results_.)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (name, grid) in zip(axes, grids.items()):
    res = grid.cv_results_
    param = list(grid.param_grid)[0]
    values = [p[param] for p in res["params"]]
    pos = values if name == "k-NN" else list(range(len(values)))   # alphas: evenly spaced
    best = int(np.argmax(res["mean_test_score"]))
    ax.plot(pos, res["mean_train_score"], marker=".", label="training")
    ax.plot(pos, res["mean_test_score"], marker=".", label="validation (5 folds)")
    ax.axvline(pos[best], color="grey", linestyle="--", label=f"best: {param} = {values[best]:g}")
    if name != "k-NN":
        ax.set_xticks(pos, [f"{v:g}" for v in values])
    ax.set_title(name)
    ax.set_xlabel("k (neighbors)" if name == "k-NN" else "alpha")
    ax.legend()
axes[0].set_ylabel("mean R²")
axes[0].set_ylim(-0.1, 1.05)
plt.show()

**What the figures show.** The gap between the two lines is overfitting: the model does much better on the listings it learned from than on new ones. *k-NN*: with $k = 1$ the training $R^2$ is a perfect 1 (every listing is its own nearest neighbor), but the validation $R^2$ is low. As $k$ grows, the two lines come together, and for large $k$ both slowly drop: underfitting. *Ridge and lasso*: with a small `alpha` they are almost plain linear regression, and the training score is clearly above the validation score. A larger `alpha` makes the weights smaller: the training score drops a little, but the validation score rises, until the best `alpha`. Beyond it, the weights are pushed so far towards 0 that both lines fall together: underfitting. Around the best settings the validation line is fairly flat, so the exact best value is partly luck of the folds. The three best scores are close (about 0.39, 0.44 and 0.45).

In [ ]:
# Ridge shrinks the weights, lasso sets some of them exactly to 0.
for name in ["Ridge", "Lasso"]:
    grid = grids[name]
    print(name)
    for alpha in grid.param_grid["alpha"]:
        w = type(grid.estimator)(alpha=alpha).fit(X, y).coef_
        print(f"  alpha = {alpha:>5g}: {np.sum(w != 0):2d} of 13 features kept, largest weight €{np.abs(w).max():3.0f}")
    best = grid.best_estimator_
    dropped = list(X.columns[best.coef_ == 0])
    print(f"  at the best alpha = {best.alpha:g}, dropped: {', '.join(dropped) if dropped else 'none'}")
    print()

In [ ]:
# The weight of every feature against alpha.
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, name in zip(axes, ["Ridge", "Lasso"]):
    grid = grids[name]
    alphas = grid.param_grid["alpha"]
    W = np.array([type(grid.estimator)(alpha=a).fit(X, y).coef_ for a in alphas])
    for j, feature in enumerate(X.columns):
        ax.plot(range(len(alphas)), W[:, j], marker=".", label=feature, color=plt.cm.tab20(j))
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xticks(range(len(alphas)), [f"{a:g}" for a in alphas])
    ax.set_xlabel("alpha")
    ax.set_ylabel("weight (€ per standard deviation)")
    ax.set_title(name)
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.show()

**What the weights show.** Ridge keeps all 13 features at every `alpha`, and makes all weights smaller together. Lasso drops features one by one as `alpha` grows, until at `alpha = 100` only `accommodates` and `bedrooms` are left, and at 300 none. At its best `alpha = 10` it drops `beds` and `superhost`, and still scores as well as ridge with all 13: `beds` says almost the same as `accommodates` and `bedrooms`, so it adds little. Note also `km_to_center`: one of the largest weights, negative (further from the center, cheaper), and it stays large for a long time in both models.

**Your turn.** Your submission. Fill in your model with its setting, a few words on how you chose it, and the $R^2$ you expect on the locked listings. The last line fits your model once more on **all** 500 training listings: when you have chosen, there is no reason to leave data out (a grid search does the same at its end, Lecture 3).

In [ ]:
best = max(grids, key=lambda name: grids[name].best_score_)
my_model = grids[best].best_estimator_
my_strategy = "5-fold cross-validation, grid search over all three models"
my_estimate = round(grids[best].best_score_, 2)

my_model.fit(X, y)     # learn from all 500 training listings

In [ ]:
# Check
check_submission()

**Think.** Before the test set is unlocked: will your model score higher or lower on it than your estimate? Why?

**Answer.** Answers differ by strategy. An estimate from the training data is far too optimistic: 1-NN even scores a perfect 1 there. An estimate from one validation split can be too high or too low, depending on which listings happened to land in the validation part. A cross-validation estimate should be close, but not exact: each fold's model learns from only four fifths of the listings, which makes the estimate a little pessimistic, and picking the best of many settings makes it a little optimistic, because the best score among many is partly luck.

> **Stop.** Wait here until the tutor unlocks the test set. Meanwhile, compare your choice with your neighbor's: did you choose the same model? The same way?

## Task 3. The test set is unlocked

**Your turn.** When the tutor shows the password on the screen, type it in quotes and run the cell. Then run the next cell: it scores your model on the 1,500 listings that no model has seen.

In [ ]:
X_test, y_test = unlock_test_set("newlistings")    # the password on the screen

In [ ]:
test_score = report(my_model, my_estimate, X_test, y_test)

Tell the tutor your model, how you chose it, your estimate and your test score. We collect them on the screen.

> **One rule from now on:** the test set has been used. If you now change your model to do better on it, its score is no longer an honest estimate for new listings (Lecture 3, "Do not choose $k$ on the test set"). The test set is for the end, once.

## Task 4. What happened?

**Run.** For a few models, the table shows four estimates side by side: the score on the training data, the score on one validation split, the mean of 5-fold cross-validation, and, in the last column, the score on the test set. Compare each estimate with the test score.

In [ ]:
check_unlocked(X_test)      # stops here while the test set is still locked
candidates = {
    "k-NN, k = 1": KNeighborsRegressor(n_neighbors=1),
    "k-NN, k = 5": KNeighborsRegressor(n_neighbors=5),
    "k-NN, k = 15": KNeighborsRegressor(n_neighbors=15),
    "k-NN, k = 50": KNeighborsRegressor(n_neighbors=50),
    "Ridge, alpha = 1": Ridge(alpha=1),
    "Ridge, alpha = 100": Ridge(alpha=100),
    "Ridge, alpha = 10000": Ridge(alpha=10000),
    "Lasso, alpha = 1": Lasso(alpha=1),
    "Lasso, alpha = 100": Lasso(alpha=100),
}
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=0)
rows = []
for name, m in candidates.items():
    rows.append({
        "model": name,
        "training": m.fit(X, y).score(X, y),
        "one split": m.fit(X_train, y_train).score(X_val, y_val),
        "5-fold CV": cross_val_score(m, X, y, cv=5).mean(),
        "test set": m.fit(X, y).score(X_test, y_test),
    })
pd.DataFrame(rows).set_index("model").round(2)

**Run.** The same for every $k$ from 1 to 50, as a plot: the score on the training data, the mean of 5-fold cross-validation, and the score on the test set.

In [ ]:
check_unlocked(X_test)
ks = range(1, 51)
train_r2, cv_r2, test_r2 = [], [], []
for k in ks:
    m = KNeighborsRegressor(n_neighbors=k)
    cv_r2.append(cross_val_score(m, X, y, cv=5).mean())
    m.fit(X, y)
    train_r2.append(m.score(X, y))
    test_r2.append(m.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ks, train_r2, label="training data")
ax.plot(ks, cv_r2, label="5-fold cross-validation")
ax.plot(ks, test_r2, label="test set", color="black")
ax.set_xlabel("k (number of neighbors)")
ax.set_ylabel("R²")
ax.legend()
plt.show()

**Think.** We discuss these together.

(a) Which model wins on the training data? How does it do on the test set? Use the words *overfitting* and *underfitting* (Lecture 3, "The sweet spot").<br>
(b) Which estimate is closest to the test score: training, one split, or cross-validation? Why?<br>
(c) Which models underfit? What happens to ridge with a huge `alpha`?<br>
(d) Suppose you had been allowed to look at the test set, and had picked the model with the best test score. Would that score be an honest estimate for next month's listings?<br>
(e) Here ridge and lasso do a little better than $k$-NN on the test set. Which feature do you think helps them most? (Hint: the one we added for you.)

**Answer.** (a) 1-NN: a perfect 1 on the training data, because every listing is its own nearest neighbor, but close to 0 on the test set (0.07): hardly better than predicting the average price. It memorized the training listings, noise included: overfitting. (b) Cross-validation: it scores every model on listings it did not learn from, and averages over five splits, so it is close to the test score (here even a little below it, because each fold learns from only 400 listings). The training score is far too optimistic, and one split can be off in either direction: it depends on which 125 listings happen to be in the validation part. Compare the estimates of your classmates who used other splits. (c) $k$-NN with a large $k$, and ridge or lasso with a large `alpha`: the model becomes so simple that it misses real patterns. Ridge with `alpha = 10000` pushes all coefficients towards 0 and predicts almost the average price, so its $R^2$ drops towards 0: underfitting. (d) No. Picking the best of many models on the test set makes its score too optimistic, as choosing $k$ on the test set in Lecture 3 did. The test set then becomes part of the choice, and you would need yet another test set to know how good the model really is. (e) `km_to_center`. A straight line cannot follow the pattern of Lab 1 in latitude and longitude (expensive in the middle, cheaper towards every edge), but in the distance to the center it is roughly a straight line: the further away, the cheaper. Lasso gives it one of its largest coefficients, about €54 less per night for every 1.7 km (one standard deviation) further from Dam Square. A good feature can help a simple model more than a cleverer model (Lecture 6).

## Check your understanding

Answer each question for yourself first, then unfold the answer. We go through them together at the end of the lab.

**1.** A student tries 40 values of $k$ and reports the best score on the test set. That score is

A. an honest estimate for new data &nbsp; B. probably too optimistic &nbsp; C. probably too pessimistic &nbsp; D. the training score

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** The best of 40 test scores is partly luck: the test set has been used to choose, so it no longer measures new data (Lecture 3, "Do not choose $k$ on the test set").

</details>

**2.** Why does 1-NN score (almost) perfectly on its training data?

A. It is the best model &nbsp; B. Every training example is its own nearest neighbor &nbsp; C. It averages many neighbors &nbsp; D. Its $R^2$ is computed differently

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** On the training data, the nearest neighbor of each listing is the listing itself, so 1-NN predicts its own price. On new listings it has no such luck.

</details>

**3.** In 10-fold cross-validation with 500 listings, how many times is a model fitted, and how many listings are in each validation part?

A. 10 times, 50 listings &nbsp; B. 5 times, 100 listings &nbsp; C. 10 times, 450 listings &nbsp; D. once, 50 listings

<details><summary><b>Answer</b> (click to unfold)</summary>

**A.** Ten folds of 50 listings; each fold is the validation part once, while the model learns from the other 450.

</details>

**4.** Ridge regression with a very large `alpha` scores an $R^2$ close to 0 on new data. Why?

A. It overfits &nbsp; B. It pushes all coefficients towards zero and predicts almost the mean &nbsp; C. It removes the outliers &nbsp; D. It uses too many features

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** With a huge penalty, all coefficients shrink towards zero, and the model predicts nearly the average price: no better than the mean, so $R^2 \approx 0$. Underfitting (Lecture 4, "Ridge regression").

</details>

**5.** Many Airbnb hosts rent out several almost identical apartments. You want a model for listings of *new* hosts. How should you split the data?

A. A random split &nbsp; B. Split by date &nbsp; C. `GroupKFold`, with the host as the group &nbsp; D. No split at all

<details><summary><b>Answer</b> (click to unfold)</summary>

**C.** With a random split, a host's twin apartments land on both sides, and the model looks better than it is: it recognizes the host. Keep all listings of one host on the same side (Lecture 3, "When a random split fails").

</details>

## If you finish early

**A. Which features does lasso keep?** Fit `Lasso(alpha=10)` on `X, y` and look at `model.coef_`, one coefficient per feature (`X.columns`). Which features does it set to zero? Do you agree with its choice?

In [ ]:
lasso = Lasso(alpha=10).fit(X, y)
print(pd.Series(lasso.coef_, index=X.columns).round(1).sort_values())
# accommodates, bedrooms and km_to_center get the largest coefficients (the further
# from the center, the cheaper). Lasso sets beds and superhost to zero, and longitude
# almost: beds adds little once accommodates and bedrooms are known, and longitude
# little once the distance to the center is known.

**B. Does the number of folds matter?** For one model, compare the mean and the spread (`scores.std()`) of the validation scores with `cv=3`, `cv=5` and `cv=10`. What changes?

In [ ]:
for folds in [3, 5, 10]:
    scores = cross_val_score(KNeighborsRegressor(n_neighbors=15), X, y, cv=folds)
    print(folds, round(scores.mean(), 3), round(scores.std(), 3))
# The mean hardly changes (about 0.37 to 0.38). With 10 folds the single scores
# vary most, because each validation part holds only 50 listings.

## Where this is going

> Judge a model only on data it has never seen. Choose with cross-validation on the training data, and open the test set once, at the end.

Today you chose a model the way it is done in practice. The strategies differed in one thing only: how honestly they estimated the score on new data.

**Coming up.**

- Lecture 5 (Monday, November 9): linear models for classification, logistic regression and linear SVMs, and their hyperparameter `C`, chosen with cross-validation, as today.
- Lecture 6 (Tuesday, November 10): preprocessing. You will put features on one scale yourself, as we did for you today, and see why the scaling, too, belongs inside the cross-validation.

**Reading.** Müller and Guido (2016), Chapter 2: "Generalization, Overfitting, and Underfitting", "k-Nearest Neighbors", "Linear Models"; Chapter 5: "Cross-Validation" and "Grid Search".

---
*This lab practices Lectures 3 and 4 of Ş. İlker Birbil; the idea of a locked test set is his. Data: [Inside Airbnb](https://insideairbnb.com), Amsterdam, collected on September 16, 2026, licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/); downloaded directly from their website and cleaned as in Lab 1.*